# Phase 1 Baseline Analysis\nThis notebook runs Locust scenarios, queries Prometheus, and generates baseline visualizations.

In [ ]:
import os
import subprocess
from datetime import datetime, timedelta

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from prometheus_api_client import PrometheusConnect

sns.set_theme(style="whitegrid")
output_dir = os.path.join(os.getcwd(), "analysis", "plots")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Install dependencies if needed
subprocess.run(
    ["python", "-m", "pip", "install", "prometheus-api-client", "pandas", "matplotlib", "seaborn", "requests"],
    check=False,
)

In [ ]:
PROM_URL = os.getenv("PROM_URL", "http://localhost:9090")
prom = PrometheusConnect(url=PROM_URL, disable_ssl=True)

def run_locust(users, spawn_rate, run_time):
    """Run a locust test with the provided parameters."""
    cmd = [
        "locust",
        "-f",
        os.path.join("load-testing", "locustfile.py"),
        "--headless",
        "-u",
        str(users),
        "-r",
        str(spawn_rate),
        "--run-time",
        run_time,
        "--host",
        "http://localhost:5000",
    ]
    return subprocess.run(cmd, check=False, capture_output=True, text=True)

def query_range(metric, start, end, step=5):
    """Query Prometheus range data for a metric."""
    result = prom.custom_query_range(metric, start_time=start, end_time=end, step=step)
    if not result:
        return pd.DataFrame()
    series = []
    for item in result:
        metric_name = item.get("metric", {}).get("service", "unknown")
        for ts, value in item.get("values", []):
            series.append({"time": datetime.fromtimestamp(ts), "value": float(value), "service": metric_name})
    return pd.DataFrame(series)

## Scenario 1: Ramp-up (0?50 users over 3 minutes, hold 2 minutes)

In [ ]:
start = datetime.utcnow()
run_locust(users=50, spawn_rate=0.28, run_time="300s")
end = datetime.utcnow()
metric = "rate(flask_http_request_total[1m]) by (service)"
df = query_range(metric, start - timedelta(seconds=30), end + timedelta(seconds=30))
if not df.empty:
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df, x="time", y="value", hue="service")
    plt.title("Ramp-up Request Rate")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "ramp_up_request_rate.png"))

## Scenario 2: Spike (10 ? 100 users for 30 seconds ? back to 10)

In [ ]:
start = datetime.utcnow()
run_locust(users=10, spawn_rate=2, run_time="60s")
run_locust(users=100, spawn_rate=10, run_time="30s")
run_locust(users=10, spawn_rate=10, run_time="60s")
end = datetime.utcnow()
metric = "rate(flask_http_request_total[1m]) by (service)"
df = query_range(metric, start - timedelta(seconds=30), end + timedelta(seconds=30))
if not df.empty:
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df, x="time", y="value", hue="service")
    plt.title("Spike Request Rate")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "spike_request_rate.png"))

## Scenario 3: Steady State (25 users for 5 minutes)

In [ ]:
start = datetime.utcnow()
run_locust(users=25, spawn_rate=2, run_time="300s")
end = datetime.utcnow()
metric = "rate(flask_http_request_total[1m]) by (service)"
df = query_range(metric, start - timedelta(seconds=30), end + timedelta(seconds=30))
if not df.empty:
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df, x="time", y="value", hue="service")
    plt.title("Steady-State Request Rate")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "steady_state_request_rate.png"))